In [0]:
%sql
WITH local_affil AS (
  SELECT station_affil, inscape_station_name, COUNT(*), COUNT(DISTINCT fk_dma_id)
  FROM detection.epg_station st
  WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
    AND st.local_or_national = 'Local'
    AND st.vendor_name = 'TIVO'
  GROUP BY 1, 2
)
, ism AS (
  SELECT ism.inscape_call_sign
  , ism.mapped_vendor
  , NVL(dma.dma_name, 'No DMA') AS dma_name
  , ism.inscape_station_name
  , ism.local_or_national
  , ism.mapped_vendor_station_id
  FROM (
    SELECT ism.inscape_station_id
    , ism.inscape_call_sign
    , st.inscape_station_name
    , ism.mapped_vendor
    , ism.mapped_vendor_station_id
    , st.fk_dma_id
    , st.station_affil
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    , CASE WHEN dp4_station_exclude.tf_station_call_sign IS NOT NULL THEN 'Local Acting as National'
           WHEN st.local_or_national = 'Local' THEN 'In Local'
           WHEN la.station_affil IS NOT NULL THEN 'In National' END AS local_or_national
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
      AND st.vendor_name = ism.mapped_vendor
    LEFT JOIN local_affil AS la
      ON la.inscape_station_name = st.inscape_station_name
     AND la.station_affil = st.station_affil
    LEFT JOIN prod.detection.nodma AS dp4_station_exclude
      ON dp4_station_exclude.tf_station_call_sign = ism.inscape_call_sign
    WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
      AND ism.mapped_vendor = 'TIVO'
      AND (la.inscape_station_name IS NOT NULL OR st.local_or_national = 'Local')
  ) ism
  LEFT JOIN prod.detection.dma
    ON ism.fk_dma_id = dma.dma_id
  WHERE ism.rn = 1
)

  SELECT ism.inscape_call_sign
  , ism.mapped_vendor_station_id
  , ism.dma_name
  , ism.inscape_station_name
  , ism.local_or_national
  , 0 AS dual_station
  , ism.inscape_station_name AS station_affil
  , 'Local Station' AS category
  FROM ism
  UNION
  SELECT na.inscape_call_sign
  , na.mapped_vendor_station_id
  , 'No DMA' AS dma_name
  , na.inscape_station_name
  , 'National' AS local_or_national
  , na.dual_station
  , na.affiliate AS station_affil
  , na.category
  FROM dev.mohit_gangwani.national_stations AS na
  WHERE na.mapped_vendor_station_id NOT IN (SELECT DISTINCT mapped_vendor_station_id FROM ism)

In [0]:
%sql
WITH local_affil AS (
  SELECT station_affil, inscape_station_name, COUNT(*), COUNT(DISTINCT fk_dma_id)
  FROM detection.epg_station st
  WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
    AND st.local_or_national = 'Local'
    AND st.vendor_name = 'TIVO'
  GROUP BY 1, 2
)
SELECT ism.*
FROM (
  SELECT ism.inscape_station_id
  , ism.inscape_call_sign
  , st.inscape_station_name
  , ism.mapped_vendor
  , ism.mapped_vendor_station_id
  , st.fk_dma_id
  , st.station_affil
  , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
  , CASE WHEN dp4_station_exclude.tf_station_call_sign IS NOT NULL THEN 'Local Acting as National'
         WHEN st.local_or_national = 'Local' THEN 'In Local'
         WHEN la.station_affil IS NOT NULL THEN 'In National' END AS local_or_national
  FROM prod.detection.inscape_station_map ism
  JOIN prod.detection.epg_station st
    ON st.station_id = ism.mapped_vendor_station_id
    AND st.vendor_name = ism.mapped_vendor
  LEFT JOIN local_affil AS la
    ON la.inscape_station_name = st.inscape_station_name
   AND la.station_affil = st.station_affil
  LEFT JOIN prod.detection.nodma AS dp4_station_exclude
    ON dp4_station_exclude.tf_station_call_sign = ism.inscape_call_sign
  WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
    AND ism.mapped_vendor = 'TIVO'
    AND (la.inscape_station_name IS NOT NULL OR st.local_or_national = 'Local')
) ism
WHERE ism.rn = 1

In [0]:
%sql
WITH inscape_station_map AS (
  SELECT ism.inscape_call_sign
  , ism.mapped_vendor
  , ism.station_affil
  , ism.inscape_station_name
  , ism.mapped_vendor_station_id
  , ism.inscape_station_id
  FROM (
    SELECT ism.inscape_station_id
    , ism.inscape_call_sign
    , st.inscape_station_name
    , ism.mapped_vendor
    , ism.mapped_vendor_station_id
    , st.fk_dma_id
    , st.station_affil
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
     AND (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
     AND st.local_or_national = 'National'
     AND st.vendor_name = 'TIVO'
  ) ism
  WHERE ism.rn = 1
)
, east_west_dual_stations AS (
  WITH dual_station AS (
    SELECT w.cleaned_station_name, COUNT(DISTINCT w.station_time_zone) AS ttl_tz, COUNT(*) AS station_count
    FROM detection.epg_station w
    JOIN inscape_station_map AS ism
      ON ism.mapped_vendor_station_id = w.station_id
     AND ism.mapped_vendor = w.vendor_name
    WHERE (w.ingested = 'TRUE' OR w.attributed = 'TRUE')
      AND w.vendor_name = 'TIVO'
      AND w.local_or_national = 'National'
  GROUP BY 1
  )
  SELECT st.cleaned_station_name
  , SPLIT_PART(st.station_time_zone, ' ', 1) AS station_tz
  , st.station_call_sign
  , st.station_id
  , ism.inscape_call_sign
  FROM prod.detection.epg_station AS st
  JOIN inscape_station_map AS ism
    ON ism.mapped_vendor_station_id = st.station_id
   AND ism.mapped_vendor = st.vendor_name
  JOIN dual_station AS ds
    ON ds.cleaned_station_name = st.cleaned_station_name
   AND ds.station_count > 1
   AND ds.ttl_tz > 1
  WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
    AND st.vendor_name = 'TIVO'
    AND st.local_or_national = 'National'
    AND st.cleaned_station_name != 'IND'
    AND st.cleaned_station_name NOT LIKE 'FanDuel%'
  GROUP BY ALL
  ORDER BY 1, 2
)

SELECT ism.*, CASE WHEN ewds.cleaned_station_name IS NOT NULL THEN 1 ELSE 0 END AS dual_station
FROM inscape_station_map AS ism
LEFT JOIN east_west_dual_stations AS ewds
ON ism.inscape_call_sign = ewds.inscape_call_sign
-- WHERE ewds.cleaned_station_name IS NULL
GROUP BY ALL

In [0]:
%sql
WITH local_affil AS (
  SELECT station_affil, inscape_station_name, COUNT(*), COUNT(DISTINCT fk_dma_id)
  FROM detection.epg_station st
  WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
    AND st.local_or_national = 'Local'
    AND st.vendor_name = 'TIVO'
  GROUP BY 1, 2
)
, ism AS (
  SELECT ism.inscape_call_sign
  , ism.mapped_vendor
  , NVL(dma.dma_name, 'No DMA') AS dma_name
  , ism.inscape_station_name
  , ism.local_or_national
  , ism.mapped_vendor_station_id
  FROM (
    SELECT ism.inscape_station_id
    , ism.inscape_call_sign
    , st.inscape_station_name
    , ism.mapped_vendor
    , ism.mapped_vendor_station_id
    , st.fk_dma_id
    , st.station_affil
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    , CASE WHEN dp4_station_exclude.tf_station_call_sign IS NOT NULL THEN 'Local Acting as National'
           WHEN st.local_or_national = 'Local' THEN 'In Local'
           WHEN la.station_affil IS NOT NULL THEN 'In National' END AS local_or_national
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
      AND st.vendor_name = ism.mapped_vendor
    LEFT JOIN local_affil AS la
      ON la.inscape_station_name = st.inscape_station_name
     AND la.station_affil = st.station_affil
    LEFT JOIN prod.detection.nodma AS dp4_station_exclude
      ON dp4_station_exclude.tf_station_call_sign = ism.inscape_call_sign
    WHERE (st.ingested = 'TRUE' OR st.attributed = 'TRUE')
      AND ism.mapped_vendor = 'TIVO'
      AND (la.inscape_station_name IS NOT NULL OR st.local_or_national = 'Local')
  ) ism
  LEFT JOIN prod.detection.dma
    ON ism.fk_dma_id = dma.dma_id
  WHERE ism.rn = 1
)
, all_stations AS (
  SELECT ism.inscape_call_sign
  , ism.mapped_vendor_station_id
  , ism.dma_name
  , ism.inscape_station_name
  , ism.local_or_national
  , 0 AS dual_station
  , ism.inscape_station_name AS station_affil
  , 'Local Station' AS category
  FROM ism
  UNION
  SELECT na.inscape_call_sign
  , na.mapped_vendor_station_id
  , 'No DMA' AS dma_name
  , na.inscape_station_name
  , 'National' AS local_or_national
  , na.dual_station
  , na.affiliate AS station_affil
  , na.category
  FROM dev.mohit_gangwani.national_stations AS na
  WHERE na.mapped_vendor_station_id NOT IN (SELECT DISTINCT mapped_vendor_station_id FROM ism)
)
SELECT DATE_TRUNC('DAY', vc.session_start) AS session_day
, CASE WHEN all_stations.inscape_station_name IS NOT NULL THEN all_stations.inscape_station_name
       WHEN all_stations.inscape_station_name IS NULL AND fk_content_id != 3468026 THEN 'Other'
       WHEN vc.tms_tuner_channel_id IS NOT NULL THEN 'Tuner'
       WHEN vc.vizio_epg_station_id IS NOT NULL THEN 'Vizio EPG'
       WHEN vc.fk_content_id = 3468026 THEN 'Null Session'
       ELSE 'XYZXYZ' END AS station_name
, CASE WHEN all_stations.dma_name IS NOT NULL THEN all_stations.dma_name
       WHEN all_stations.dma_name IS NULL AND fk_content_id != 3468026 THEN 'Other'
       WHEN vc.tms_tuner_channel_id IS NOT NULL THEN 'Tuner'
       WHEN vc.vizio_epg_station_id IS NOT NULL THEN 'Vizio EPG'
       WHEN vc.fk_content_id = 3468026 THEN 'Null Session'
       ELSE 'XYZXYZ' END AS dma_name
, CASE WHEN all_stations.local_or_national IS NOT NULL THEN all_stations.local_or_national
       WHEN all_stations.local_or_national IS NULL AND fk_content_id != 3468026 THEN 'Other'
       WHEN vc.tms_tuner_channel_id IS NOT NULL THEN 'Tuner'
       WHEN vc.vizio_epg_station_id IS NOT NULL THEN 'Vizio EPG'
       WHEN vc.fk_content_id = 3468026 THEN 'Null Session'
       ELSE 'XYZXYZ' END AS local_or_national
, COALESCE(all_stations.dual_station, 0) AS dual_station
, CASE WHEN all_stations.station_affil IS NOT NULL THEN all_stations.station_affil
       WHEN all_stations.station_affil IS NULL AND fk_content_id != 3468026 THEN 'Other'
       WHEN vc.tms_tuner_channel_id IS NOT NULL THEN 'Tuner'
       WHEN vc.vizio_epg_station_id IS NOT NULL THEN 'Vizio EPG'
       WHEN vc.fk_content_id = 3468026 THEN 'Null Session'
       ELSE 'XYZXYZ' END AS station_affil
, CASE WHEN all_stations.category IS NOT NULL THEN all_stations.category
       WHEN all_stations.category IS NULL AND fk_content_id != 3468026 THEN 'Other'
       WHEN vc.tms_tuner_channel_id IS NOT NULL THEN 'Tuner'
       WHEN vc.vizio_epg_station_id IS NOT NULL THEN 'Vizio EPG'
       WHEN vc.fk_content_id = 3468026 THEN 'Null Session'
       ELSE 'XYZXYZ' END AS category
, COUNT(*)*1.0 AS sessions_count
, SUM(vc.session_duration)/3600.0 AS total_duration
, COUNT(Distinct vc.fk_tvid) AS tv_count
FROM prod.detection.viewing_content_firehose vc
LEFT JOIN all_stations AS all_stations
  ON all_stations.mapped_vendor_station_id = vc.fk_station_id
WHERE vc.session_start >= '2025-02-26 00:00:00'
  AND vc.session_start < '2025-04-14 00:00:00'
  AND vc.fk_zoo_id = 17
  AND vc.partition_key >= '2025-02-26'
  AND vc.partition_key < '2025-04-14'
GROUP BY 1, 2, 3, 4, 5, 6, 7

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.issue_with_innovid_foxnews_20250417 AS
WITH all_stations AS (
  SELECT na.inscape_call_sign
  , na.mapped_vendor_station_id
  , na.inscape_station_name
  , na.dual_station
  , na.affiliate AS station_affil
  , na.category
  FROM dev.mohit_gangwani.national_stations AS na
  WHERE na.category LIKE 'News%'
)
, main_stats AS (
  SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
  , vc.is_live
  --, all_stations.inscape_call_sign
  , all_stations.inscape_station_name AS station_name
  , all_stations.category AS category
  , COUNT(*)*1.0 AS session_count
  , SUM(vc.session_duration)/3600.0 AS total_duration
  , COUNT(DISTINCT vc.fk_tvid) AS tv_count
  FROM prod.detection.viewing_content_firehose vc
  JOIN all_stations AS all_stations
    ON all_stations.mapped_vendor_station_id = vc.fk_station_id
  WHERE vc.session_start >= '2024-11-01 00:00:00'
    AND vc.session_start < '2025-03-01 00:00:00'
    AND vc.fk_zoo_id = 17
    AND vc.partition_key >= '2024-11-01'
    AND vc.partition_key < '2025-03-01'
    AND vc.fk_content_id != 3468026
  GROUP BY 1, 2, 3, 4
)
, norm_vals AS (
  SELECT is_live
  , station_name
  --, inscape_call_sign
  , AVG(session_count*1.0)     AS avg_sessions
  , AVG(tv_count*1.0)          AS avg_tvs
  , AVG(total_duration*1.0)    AS avg_duration
  , STDDEV(session_count*1.0)  AS stdev_sessions
  , STDDEV(tv_count*1.0)       AS stdev_tvs
  , STDDEV(total_duration*1.0) AS stdev_duration
  FROM main_stats
  WHERE tv_count > 0
  GROUP BY 1, 2
)
SELECT ms.*
, (ms.session_count-nm.avg_sessions)/(nm.stdev_sessions*1.0)  AS normalized_session_count
, (ms.tv_count-nm.avg_tvs)/(nm.stdev_tvs)                     AS normalized_tv_count
, (ms.total_duration-nm.avg_duration)/(nm.stdev_duration*1.0) AS normalized_total_duration
FROM main_stats ms
JOIN norm_vals nm
  ON ms.is_live = nm.is_live
 AND ms.station_name = nm.station_name
--  AND ms.inscape_call_sign = nm.inscape_call_sign

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_wo_live AS
WITH all_stations AS (
  SELECT na.inscape_call_sign
  , na.mapped_vendor_station_id
  , na.inscape_station_name
  , na.dual_station
  , na.affiliate AS station_affil
  , na.category
  FROM dev.mohit_gangwani.national_stations AS na
  WHERE na.category LIKE 'News%'
)
, main_stats AS (
  SELECT DATE_TRUNC('HOUR', vc.session_start) AS session_hour
  , all_stations.inscape_station_name AS station_name
  , all_stations.category AS category
  , COUNT(*)*1.0 AS session_count
  , SUM(vc.session_duration)/3600.0 AS total_duration
  , COUNT(DISTINCT vc.fk_tvid) AS tv_count
  FROM prod.detection.viewing_content_firehose vc
  JOIN all_stations AS all_stations
    ON all_stations.mapped_vendor_station_id = vc.fk_station_id
  WHERE vc.session_start >= '2024-11-01 00:00:00'
    AND vc.session_start < '2025-03-01 00:00:00'
    AND vc.fk_zoo_id = 17
    AND vc.partition_key >= '2024-11-01'
    AND vc.partition_key < '2025-03-01'
    AND vc.fk_content_id != 3468026
  GROUP BY 1, 2, 3
)
, norm_vals AS (
  SELECT station_name
  , AVG(session_count*1.0)     AS avg_sessions
  , AVG(tv_count*1.0)          AS avg_tvs
  , AVG(total_duration*1.0)    AS avg_duration
  , STDDEV(session_count*1.0)  AS stdev_sessions
  , STDDEV(tv_count*1.0)       AS stdev_tvs
  , STDDEV(total_duration*1.0) AS stdev_duration
  FROM main_stats
  WHERE tv_count > 0
  GROUP BY 1
)
SELECT ms.*
, (ms.session_count-nm.avg_sessions)/(nm.stdev_sessions*1.0)  AS normalized_session_count
, (ms.tv_count-nm.avg_tvs)/(nm.stdev_tvs)                     AS normalized_tv_count
, (ms.total_duration-nm.avg_duration)/(nm.stdev_duration*1.0) AS normalized_total_duration
FROM main_stats ms
JOIN norm_vals nm
  ON ms.station_name = nm.station_name

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live AS
WITH all_stations AS (
  SELECT na.inscape_call_sign
  , na.mapped_vendor_station_id
  , na.inscape_station_name
  , na.dual_station
  , na.affiliate AS station_affil
  , na.category
  FROM dev.mohit_gangwani.national_stations AS na
  WHERE na.category LIKE 'News%'
)
, main_stats AS (
  SELECT DATE(vc.session_start) AS session_day
  , all_stations.inscape_station_name AS station_name
  , all_stations.category AS category
  , COUNT(*)*1.0 AS session_count
  , SUM(vc.session_duration)/3600.0 AS total_duration
  , COUNT(DISTINCT vc.fk_tvid) AS tv_count
  FROM prod.detection.viewing_content_firehose vc
  JOIN all_stations AS all_stations
    ON all_stations.mapped_vendor_station_id = vc.fk_station_id
  WHERE vc.session_start >= '2024-11-01 00:00:00'
    AND vc.session_start < '2025-03-01 00:00:00'
    AND vc.fk_zoo_id = 17
    AND vc.partition_key >= '2024-11-01'
    AND vc.partition_key < '2025-03-01'
    AND vc.fk_content_id != 3468026
  GROUP BY 1, 2, 3
)
, norm_vals AS (
  SELECT station_name
  , AVG(session_count*1.0)     AS avg_sessions
  , AVG(tv_count*1.0)          AS avg_tvs
  , AVG(total_duration*1.0)    AS avg_duration
  , STDDEV(session_count*1.0)  AS stdev_sessions
  , STDDEV(tv_count*1.0)       AS stdev_tvs
  , STDDEV(total_duration*1.0) AS stdev_duration
  FROM main_stats
  WHERE tv_count > 0
  GROUP BY 1
)
SELECT ms.*
, (ms.session_count-nm.avg_sessions)/(nm.stdev_sessions*1.0)  AS normalized_session_count
, (ms.tv_count-nm.avg_tvs)/(nm.stdev_tvs)                     AS normalized_tv_count
, (ms.total_duration-nm.avg_duration)/(nm.stdev_duration*1.0) AS normalized_total_duration
FROM main_stats ms
JOIN norm_vals nm
  ON ms.station_name = nm.station_name

In [0]:
%sql
WITH fox_news AS (
  SELECT session_day, station_name, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name = 'Fox News Channel'
    AND session_day <= '2025-02-24'
)
, non_fox_news AS (
  SELECT session_day, station_name, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name != 'Fox News Channel'
    AND session_day <= '2025-02-24'
)
SELECT f.station_name
, SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))/COUNT(DISTINCT nf.session_day) AS mae
, (SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))*SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration)))/COUNT(DISTINCT nf.session_day) AS mse
, REGR_R2(f.normalized_total_duration, nf.normalized_total_duration) AS r2
, CORR(f.normalized_total_duration, nf.normalized_total_duration) AS correlation
FROM fox_news nf
JOIN non_fox_news f
  ON nf.session_day = f.session_day
GROUP BY 1

In [0]:
%sql
WITH fox_news AS (
  SELECT SUM(total_duration) AS fox_news_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name = 'Fox News Channel'
    AND session_day <= '2025-02-24'
)
, non_fox_news AS (
  SELECT SUM(total_duration) AS newsmax_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name = 'Newsmax in'
    AND session_day <= '2025-02-24'
)
SELECT fn.fox_news_total_duration
, nn.newsmax_total_duration
, fn.fox_news_total_duration/nn.newsmax_total_duration
FROM fox_news fn
, non_fox_news nn

In [0]:
%sql
WITH fox_news AS (
  SELECT SUM(total_duration) AS fox_news_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name = 'Fox News Channel'
)
, non_fox_news AS (
  SELECT SUM(total_duration) AS newsmax_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_tuner_only
)
SELECT fn.fox_news_total_duration
, nn.newsmax_total_duration
, fn.fox_news_total_duration/nn.newsmax_total_duration
FROM fox_news fn
, non_fox_news nn

In [0]:
'''
Scaling Factor for Newsmax: 13.448612
Scaling Factor For Tuner: 457.282669
'''

In [0]:
%sql
SELECT DATE_TRUNC('DAY', ts.session_start) AS session_day
, (SUM(TIMESTAMPDIFF(SECOND, ts.session_start, ts.session_end))/3600.0)*457.282669 AS total_duration
FROM detection.tuner_sessionized ts
WHERE session_start >= '2025-03-01 00:00:00'
  AND session_start < '2025-04-08 00:00:00'
  AND ts.tuner_channel_name LIKE 'FNC%'
  AND ts.callsign_tv2 IS NULL
GROUP BY 1

In [0]:
%sql
SELECT * FROM detection.epg_station
WHERE vendor_name = 'TIVO'
AND inscape_station_name LIKE 'Newsmax%'

In [0]:
%sql
SELECT DATE(vc.session_start) AS session_day
, (SUM(vc.session_duration)/3600.0)*13.448612 AS total_duration
FROM prod.detection.viewing_content_firehose vc
WHERE vc.session_start >= '2025-03-01 00:00:00'
  AND vc.session_start < '2025-04-08 00:00:00'
  AND vc.fk_zoo_id = 17
  AND vc.partition_key >= '2025-03-01'
  AND vc.partition_key < '2025-04-08'
  AND vc.fk_content_id != 3468026
  AND vc.fk_station_id = 90856
GROUP BY 1

In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_tuner_only AS
WITH main_stats AS (
  SELECT DATE_TRUNC('DAY', ts.session_start) AS session_day
  , COUNT(*) AS session_count
  , SUM(TIMESTAMPDIFF(SECOND, ts.session_start, ts.session_end))/3600.0 AS total_duration
  , COUNT(DISTINCT ts.tvid) AS tv_count
  FROM detection.tuner_sessionized ts
  WHERE session_start >= '2024-11-01 00:00:00'
    AND session_start < '2025-03-01 00:00:00'
    AND ts.tuner_channel_name LIKE 'FNC%'
    AND ts.callsign_tv2 IS NULL
  GROUP BY 1
)
, norm_vals AS (
  SELECT AVG(session_count*1.0)   AS avg_sessions
  , AVG(tv_count*1.0)             AS avg_tvs
  , AVG(total_duration*1.0)       AS avg_duration
  , STDDEV(session_count*1.0)     AS stdev_sessions
  , STDDEV(tv_count*1.0)          AS stdev_tvs
  , STDDEV(total_duration*1.0)    AS stdev_duration
  FROM main_stats
)
SELECT ms.*
, (ms.session_count-nm.avg_sessions)/(nm.stdev_sessions*1.0)  AS normalized_session_count
, (ms.tv_count-nm.avg_tvs)/(nm.stdev_tvs)                     AS normalized_tv_count
, (ms.total_duration-nm.avg_duration)/(nm.stdev_duration*1.0) AS normalized_total_duration
FROM main_stats ms, norm_vals nm

In [0]:
%sql
WITH fox_news AS (
  SELECT session_day, station_name, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
  WHERE station_name = 'Fox News Channel'
)
, tuner AS (
  SELECT session_day, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_tuner_only
)
SELECT SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))/COUNT(DISTINCT f.session_day) AS mae
, (SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))*SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration)))/COUNT(DISTINCT f.session_day) AS mse
, REGR_R2(f.normalized_total_duration, nf.normalized_total_duration) AS r2
, CORR(f.normalized_total_duration, nf.normalized_total_duration) AS correlation
FROM fox_news nf
JOIN tuner f
  ON nf.session_day = f.session_day

In [0]:
%sql
WITH fox_news AS (
  SELECT session_hour, station_name, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_wo_live
  WHERE station_name = 'Fox News Channel'
    AND session_hour < '2025-02-25 00:00:00'
)
, non_fox_news AS (
  SELECT session_hour, station_name, normalized_total_duration
  FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_wo_live
  WHERE station_name != 'Fox News Channel'
    AND session_hour < '2025-02-25 00:00:00'
)
SELECT f.station_name
, SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))/COUNT(DISTINCT f.session_hour) AS mae
, (SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration))*SUM(ABS(f.normalized_total_duration-nf.normalized_total_duration)))/COUNT(DISTINCT f.session_hour) AS mse
, REGR_R2(f.normalized_total_duration, nf.normalized_total_duration) AS r2
FROM fox_news nf
JOIN non_fox_news f
  ON nf.session_hour = f.session_hour
GROUP BY 1

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime, date

In [0]:
df_pre = spark.sql(
    f"""
    SELECT DATE(vc.session_start) AS session_day
    , SUM(vc.session_duration)/3600.0 AS ttl_duration
    FROM detection.viewing_content_firehose vc
    WHERE vc.session_start >= '2024-08-01 00:00:00'
    AND vc.session_start < '2025-04-18 00:00:00'
    AND vc.fk_zoo_id = 17
    AND vc.fk_station_id IN (136479, 129309, 92150)
    AND vc.partition_key >= '2024-08-01'
    AND vc.session_start < '2025-04-18'
    GROUP BY 1
 """)

In [0]:
df = df_pre.toPandas().fillna(0)

In [0]:
df.sort_values(by='session_day', inplace=True)

In [0]:
df.reset_index(drop=True, inplace=True)

In [0]:
df.head()

In [0]:
df['session_day'] = pd.to_datetime(df['session_day'])

In [0]:
ndf = df.copy()

In [0]:
df = ndf.copy()

In [0]:
df.loc[:, 'special_event'] = 0

df.loc[:, 'special_event'] = np.where(df.session_day == '2024-11-06', 3, df.special_event)
df.loc[:, 'special_event'] = np.where(df.session_day == '2025-01-20', 2, df.special_event)
df.loc[:, 'special_event'] = np.where(df.session_day == '2025-03-05', 1, df.special_event)

In [0]:
df.loc[df['session_day'].dt.strftime('%Y-%m-%d').between('2025-03-02', '2025-04-09'), 'ttl_duration'] = 0

In [0]:
df.loc[df['session_day'].dt.strftime('%Y-%m-%d').between('2024-12-16', '2025-01-06'), 'ttl_duration'] = 0

In [0]:
# df[df['session_day'].dt.strftime('%Y-%m-%d').between('2025-03-01', '2025-04-08')]

In [0]:
df.set_index('session_day', inplace=True)

df['dayofweek'] = df.index.dayofweek
df['day'] = df.index.day
df['month'] = df.index.month

df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

In [0]:
df_known = df[df['ttl_duration'] > 0]
df_missing = df[df['ttl_duration'] == 0]

features = ['dayofweek', 'day', 'month', 'dow_sin', 'dow_cos', 'special_event']
X_train = df_known[features]
y_train = df_known['ttl_duration']

X_missing = df_missing[features]

In [0]:
model = GradientBoostingRegressor(n_estimators=100, max_depth=7, random_state=42)
model.fit(X_train, y_train)

In [0]:
y_pred = model.predict(X_missing)
df.loc[df_missing.index, 'ttl_duration_predicted'] = y_pred

In [0]:
df.head()

In [0]:
df['ttl_duration'] = pd.to_numeric(df['ttl_duration'], errors='coerce')
df['ttl_duration_predicted'] = pd.to_numeric(df['ttl_duration_predicted'], errors='coerce')

In [0]:
plt.figure(figsize=(15,5))
df['ttl_duration'].plot(label='Known', alpha=0.6)
df['ttl_duration_predicted'].plot(label='Predicted (Long Gap)', linestyle='--')
plt.legend()
plt.title('Total Duration for Fox News: Forecast for Missing Month')
plt.show()

In [0]:
pred_df = df[df['ttl_duration_predicted'].notnull()]

In [0]:
pred_df.drop(columns=['ttl_duration', 'dayofweek', 'day', 'month', 'dow_sin', 'dow_cos'], inplace=True)

In [0]:
pred_df.reset_index(inplace=True)

In [0]:
pred_df

In [0]:
spark_df = spark.createDataFrame(pred_df)

In [0]:
spark_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dev.mohit_gangwani.fox_news_predicted_values_for_missing_period_innovid")

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
limit 100